In [ ]:
import cv2
import mediapipe as mp
import time

In [ ]:
class handDetector():
    def __init__(self, mode=False, maxHands=1, modelComplexity=1, detectionCon=0.75, trackCon=0.5):
        self.mode = mode
        self.maxHands = maxHands
        self.modelComplex = modelComplexity
        self.detectionCon = detectionCon
        self.trackCon = trackCon

        self.mpHands = mp.solutions.hands
        self.hands = self.mpHands.Hands(self.mode, self.maxHands, self.modelComplex, 
                                        self.detectionCon, self.trackCon)
        self.mpDraw = mp.solutions.drawing_utils
    def findHands(self, img, draw=True):
        imgRGB = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        self.results = self.hands.process(imgRGB)
        #print(results.multi_hand_landmarks)
 
        if self.results.multi_hand_landmarks:
            for handLms in self.results.multi_hand_landmarks:
                if draw:
                    self.mpDraw.draw_landmarks(img, handLms,
                                               self.mpHands.HAND_CONNECTIONS)
        return img
 
    def findPosition(self, img, handNo=0, draw=True):
 
        lmList = []
        if self.results.multi_hand_landmarks:
            myHand = self.results.multi_hand_landmarks[handNo]
            for id, lm in enumerate(myHand.landmark):
                # print(id, lm)
                h, w, c = img.shape
                cx, cy = int(lm.x * w), int(lm.y * h)
                # print(id, cx, cy)
                lmList.append([id, cx, cy])
                if draw:
                    cv2.circle(img, (cx, cy), 15, (255, 0, 255), cv2.FILLED)
 
        return lmList

In [ ]:
#setting numbers for each gestures
def getNumber(ar):
    s = ""
    for i in ar:
        s += str(ar[i]);

    if(s == "00000"):
        return (0)
    elif(s == "01000"):
        return(1)
    elif(s == "01100"):
        return(2)
    elif(s == "01110"):
        return(3)
    elif(s == "01111"):
        return(4)
    elif(s == "11111"):
        return(5)
    elif(s == "01001"):
        return(6)
    elif(s == "01011"):
        return(7)

In [ ]:
import cv2
pTime = 0
cTime = 0
cv2.namedWindow("preview")

wcam, hcam = 640, 480
vc = cv2.VideoCapture(0)
vc.set(3, wcam)
vc.set(4, hcam)

detector = handDetector()
if vc.isOpened(): # try to get the first frame
    rval, img = vc.read()
else:
    rval = False

while rval:
    cv2.imshow("preview", img)
    rval, img = vc.read()
    img = detector.findHands(img)
    lmList = detector.findPosition(img)
    if len(lmList) != 0:
            print(lmList[4])
    tipId = [4, 8, 12, 16, 20]
#     tipId = [1, 10, 20, 30, 40]
    if(len(lmList) != 0):
        fingers = []
        # thumb
        if(lmList[tipId[0]][1] > lmList[tipId[0]-1][1]):
            fingers.append(1)
        else:
            fingers.append(0)
        # 4 fingers
        for id in range(1, len(tipId)):

            if(lmList[tipId[id]][2] < lmList[tipId[id]-2][2]):
                fingers.append(1)

            else:
                fingers.append(0)

        #draw landmarks,text for each gestures
        cv2.rectangle(img, (20, 255), (170, 425), (0, 255, 0), cv2.FILLED)
        cv2.putText(img, str(getNumber(fingers)), (45, 375), cv2.FONT_HERSHEY_PLAIN,
                                     10, (255, 0, 0), 20)
            
            
            
    key = cv2.waitKey(20)
    if key == 27: # exit on ESC
        break

vc.release()
cv2.destroyWindow("preview")